# SQL query from table names

In This notebook we are going to test if using just the name of the table, and a shord definition of its contect we can use a model like GTP3.5-Turbo to select which tables are necessary to create a SQL Order to answer the user petition.

In [2]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [3]:
#Functio to call the model.
def return_OAI(user_message):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)
    context = []
    context.append({'role':'system', "content": user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=context,
            temperature=0,
        )

    return (response.choices[0].message.content)

In [7]:
#Definition of the tables.
import pandas as pd

# Table and definitions sample
data = {'table': [
    "customers(id, name, email)", 
    "orders(id, customer_id, order_date, total_amount)"
    ],
        'definition': [
            "Basic customer info.",
            "Orders placed by customers; total_amount is the order total."
        ]}
df = pd.DataFrame(data)
print(df)

                                               table  \
0                         customers(id, name, email)   
1  orders(id, customer_id, order_date, total_amount)   

                                          definition  
0                               Basic customer info.  
1  Orders placed by customers; total_amount is th...  


In [8]:
text_tables = '\n'.join([f"{row['table']}: {row['definition']}" for index, row in df.iterrows()])

In [9]:
print(text_tables)

customers(id, name, email): Basic customer info.
orders(id, customer_id, order_date, total_amount): Orders placed by customers; total_amount is the order total.


In [10]:
prompt_question_tables = """
Given the following tables and their content definitions,
###Tables
{tables}

Tell me which tables would be necessary to query with SQL to address the user's question below.
Return the table names in a json format.
###User Questyion:
{question}
"""


In [11]:
#Creating the prompt, with the user questions and the tables definitions.
pqt1 = prompt_question_tables.format(tables=text_tables, question="Which customers spent the most in the last 90 days?")

In [12]:
print(return_OAI(pqt1))

{
    "tables": ["customers", "orders"]
}


In [13]:
pqt3 = prompt_question_tables.format(tables=text_tables,
                                     question="Show total revenue per month in 2025.")

In [14]:
print(return_OAI(pqt3))

```json
{
    "tables": ["orders"]
}
```


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try a few versions if you have time
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

### Version 1 — Strict table selection (JSON only)
Pick only the tables needed. If unsure, return an empty list.

In [15]:
question_1 = "Which customers spent the most in the last 90 days? Return top 5."

pqt1 = prompt_question_tables.format(
    tables=text_tables,
    question=question_1
)

print(return_OAI(pqt1))

{
    "tables": ["customers", "orders"]
}


### Version 2 — Minimal tables + quick reason
Pick the smallest set of tables and explain why each one is needed.

In [20]:
question_2 = "How many orders were placed per month in 2025?"

pqt2 = f"""
{prompt_question_tables.format(tables=text_tables, question=question_2)}

Return JSON only:
{{
  "tables": ["..."],
  "why": {{"table_name":"short reason"}}
}}
"""
print(return_OAI(pqt2))

{
  "tables": ["orders"],
  "why": {"orders":"Contains information about orders and order dates"}
}


### Version 3 — Tables + join path + SQL sketch
Pick tables, suggest likely join keys, and draft a short SQL skeleton.

In [21]:
question_3 = "Show total revenue per customer this year. Include customer name."

pqt3 = f"""
{prompt_question_tables.format(tables=text_tables, question=question_3)}

Return JSON only:
{{
  "tables": ["..."],
  "join_path": ["table_a.col = table_b.col", "..."],
  "sql_skeleton": "SELECT ... FROM ... JOIN ... WHERE ... GROUP BY ...;"
}}
"""
print(return_OAI(pqt3))

{
  "tables": ["customers", "orders"],
  "join_path": ["customers.id = orders.customer_id"],
  "sql_skeleton": "SELECT customers.name, SUM(orders.total_amount) AS total_revenue FROM customers JOIN orders ON customers.id = orders.customer_id WHERE YEAR(orders.order_date) = YEAR(CURDATE()) GROUP BY customers.id;"
}


## Summary
I created 3 prompt versions to test how well the model can choose relevant SQL tables from schema definitions.
The main difference between versions was the required output detail: tables only (V1) → tables + reason (V2) → tables + join + SQL sketch (V3).

## Findings (evidence)
**V1 output:** `{"tables": ["customers", "orders"]}`
- This matches the question because “top spenders” needs spend totals from `orders` and the customer identity from `customers`.

**V2 output:** `{"tables": ["orders"], "why": {"orders":"Contains information about orders and order dates"}}`
- This is correctly minimal: counting orders per month only needs `orders` (date is in this table). `customers` would be extra.

**V3 output:** `{"tables":["customers","orders"], "join_path":["customers.id = orders.customer_id"], ...}`
- The join key is logically consistent with the schema and shows the model can infer table relationships.
- The SQL sketch matches the goal: aggregate (`SUM`) per customer + include name.

## What I learned
- Requiring a **"why"** field helps validate table choice quickly (V2 made it obvious why only `orders` was needed).
- Asking for a **join_path** exposes assumptions about foreign keys (V3 correctly inferred `customers.id = orders.customer_id`).
- More structured prompting = easier debugging: I can spot missing tables/keys immediately instead of guessing from a plain list.

## Note
The SQL sketch in V3 uses `YEAR(order_date)`, which may vary by SQL dialect (e.g., SQLite typically uses `strftime('%Y', order_date)`).